# ABSA Pipeline — LoRA M1 + Binary Sentiment M2
## Architecture
```
final.csv (2400+ labeled)  ──► Iterative Stratified Split
                                    ├── Train (~70%)
                                    ├── Val   (~15%)  ← early stopping / threshold tuning
                                    └── Test  (~15%)  ← LOCKED, open once

final_nonlabel.csv (1.6M)  ──► Filter English + Remove overlap ──► Unlabeled Pool

STEP 1 — DAPT
  roberta-base + MLM on (Train sentences + Unlabeled Pool) → Domain-Adapted RoBERTa

STEP 2 — Multi-task Model 1  [LoRA rank=32]
  Domain-Adapted RoBERTa + LoRA
    ├── Head 1: Related/Unrelated     (all sentences,  BCE, weight 0.2)
    ├── Head 2: General/Technical     (related only,   BCE, weight 0.2)
    └── Head 3: Aspect (N-class MLB)  (technical only, BCE, weight 0.6)

STEP 3 — Sentiment Model 2  [Binary Neg/Pos] (due to the low number of Neutral)
  yangheng/deberta-v3-base-absa-v1.1
```

In [1]:
!pip install scikit-multilearn -q
!pip install langdetect deep-translator -q
!pip install nlpaug -q
!pip install peft -q
!pip install -U torchao peft transformers accelerate

^C



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 17.5 MB/s eta 0:00:00
Using cached peft-0.19.1-py3-none-any.whl (680 kB)
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ------------------------------------- -- 10.0/10.6 MB 50.5 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 38.8 MB/s eta 0:00:00

   ---------------------------------------- 0/3 [torchao]
   ---------------------------------------- 0/3 [torchao]
   ---------------------------------------- 0/3 [torchao]
   ---------------------------------------- 0/3 [torchao]
   ---------------------------------------- 0/3 [torchao]
   ---------------------------------------- 0/3 [torchao]
   ---------------------------------------- 0/3 [torchao]
   ---------------------------------------- 0/3 [torchao]
   ---------------------------------------- 0/3 [


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from datasets import Dataset as HFDataset

from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup,
)

from peft import LoraConfig, get_peft_model

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from skmultilearn.model_selection import iterative_train_test_split

import nlpaug.augmenter.word as naw
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
import uuid
import warnings, os, json, re

warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score, fbeta_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

Device: cuda
GPU: NVIDIA H100 80GB HBM3


In [ ]:
class CFG:
    # ── Paths ──
    LABELED_PATH    = 'final.csv'
    UNLABELED_PATH  = 'final_nonlabel.csv'
    TEXT_COL        = 'sentence_for_model'
    ASPECT_COL      = 'target_aspect'
    SENTIMENT_COL   = 'target_sentiment'
    IS_RELATED_COL  = 'is_related'
    IS_TECH_COL     = 'is_technical'
    ROW_ID_COL      = 'row_id'

    # ── Models ──
    BACKBONE        = 'roberta-base'
    SENTIMENT_MODEL = 'yangheng/deberta-v3-base-absa-v1.1'
    DAPT_OUTPUT     = './dapt_roberta'
    M1_OUTPUT       = './model1_multitask'
    M2_OUTPUT       = './model2_sentiment'

    # ── Split ──
    SEED            = 42
    VAL_RATIO       = 0.15
    TEST_RATIO      = 0.15

    # ── DAPT ──
    DAPT_MAX_LEN    = 128
    DAPT_BATCH      = 32
    DAPT_EPOCHS     = 4
    DAPT_LR         = 5e-5
    DAPT_UNLABELED_SAMPLE = 300_000

    # ── Model 1 ──
    M1_MAX_LEN      = 128
    M1_BATCH        = 16
    M1_EPOCHS       = 50
    M1_LR           = 5e-5
    M1_WEIGHT_DECAY = 0.05
    M1_WARMUP_RATIO = 0.1
    LOSS_W_H1       = 0.2
    LOSS_W_H2       = 0.2
    LOSS_W_H3       = 0.6
    USE_UNCERTAINTY_WEIGHTS = False
    M1_PATIENCE     = 4
    M1_MONITOR_METRIC = 'f1_mean'   # 'f1_mean' | 'f1_h3' | 'f1_h1' ...
    FREEZE_H1_AFTER_EPOCH = 10
    AUG_RARE_THRESHOLD    = 150
    AUG_NUM_PER_SAMPLE    = 2

    # ── LORA ──
    M1_LORA_R       = 32
    M1_LORA_ALPHA   = 32 #
    M1_LORA_DROPOUT = 0.1
    M1_LORA_TARGET  = ['query', 'key', 'value', 'dense']

    # ── Model 2 ──
    M2_MAX_LEN      = 128
    M2_BATCH        = 30
    M2_EPOCHS       = 10
    M2_LR           = 2e-5
    M2_PATIENCE     = 3
    FORCE_RETRAIN_M2 = False  # set True to skip checkpoint and train again for Model 2

    # ── Inference ──
    UNSURE_LABEL    = 'unsure'
    M2_INFERENCE_BATCH = 64

torch.manual_seed(CFG.SEED)
np.random.seed(CFG.SEED)
print('Config loaded. hehehe')

Config loaded. hehehe


## 2. Data Loading & Preparation
Load labeled data, assign unique UUIDs to rows, and extract valid aspect-sentiment pairs while excluding non-target classes.

Load labeled data and assign unique UUIDs to rows.

In [ ]:
df = pd.read_csv(CFG.LABELED_PATH).dropna(subset=[CFG.TEXT_COL])
print(f'Labeled rows: {len(df)}')
print('Columns:', df.columns.tolist())

# row_id UUID
df[CFG.ROW_ID_COL] = [str(uuid.uuid4()) for _ in range(len(df))]
df = df.reset_index(drop=True)
print(f'row_id sample: {df[CFG.ROW_ID_COL].iloc[0]}')
print('row_id unique check:', df[CFG.ROW_ID_COL].is_unique)  # Need to be True

Labeled rows: 2410
Columns: ['original_id', 'parent_asin', 'product_title', 'rating', 'review_context', 'sentence', 'sentence_for_model', 'target_aspect', 'target_sentiment', 'is_related', 'is_technical']
row_id sample: 0fe6d946-ca07-4bc1-aa95-8108fa9b3668
row_id unique check: True


Parse & encode aspects

In [ ]:
EXCLUDE_ASPECTS = {'general', 'unrelated'}

def parse_and_validate_pairs(aspect_str, sentiment_str):
    """
    Parse Aspect and Sentiment.
    Drop short rows -> consistency in M1 and M2 data
    """
    if pd.isna(aspect_str) or pd.isna(sentiment_str):
        return [], []

    raw_aspects = [a.strip().lower() for a in str(aspect_str).split(';')]
    raw_sents   = [s.strip() for s in str(sentiment_str).split(';')]

    if len(raw_aspects) != len(raw_sents):
        return [], []

    valid_aspects, valid_sents = [], []
    for asp, sent in zip(raw_aspects, raw_sents):
        if asp and asp not in EXCLUDE_ASPECTS:
            try:
                sent_int = int(float(sent))
                valid_aspects.append(asp)
                valid_sents.append(sent_int)
            except ValueError:
                pass

    return valid_aspects, valid_sents

Apply Parser

In [ ]:
parsed_results = df.apply(
    lambda x: parse_and_validate_pairs(x[CFG.ASPECT_COL], x[CFG.SENTIMENT_COL]),
    axis=1
)
df['aspects_parsed'] = parsed_results.apply(lambda x: x[0])
df['sentiments_parsed'] = parsed_results.apply(lambda x: x[1])
df['is_valid_pairs'] = df['aspects_parsed'].apply(len) > 0

## 3. Iterative Stratified Split (multilabel-safe)
Perform iterative stratified splitting on aspect-sentiment combinations to create balanced train/val/test sets, fit the binarizer without leakage, and merge non-technical rows.

Only valid, related, and technical rows are extracted

In [ ]:
df_tech = df[
    (df[CFG.IS_RELATED_COL] == 1) &
    (df[CFG.IS_TECH_COL] == 1) &
    (df['is_valid_pairs'] == True)
].copy().reset_index(drop=True)

Create combo labels aspect_sentiment for iterative stratified split

In [ ]:
def get_combo_labels(aspects, sentiments):
    return [f"{a}_{s}" for a, s in zip(aspects, sentiments)]

In [ ]:
df_tech['combo_labels'] = df_tech.apply(
    lambda x: get_combo_labels(x['aspects_parsed'], x['sentiments_parsed']),
    axis=1
)

Convert combo labels into multi-hot binary

In [ ]:
mlb_combo = MultiLabelBinarizer()
y_combo = np.array(mlb_combo.fit_transform(df_tech['combo_labels']), dtype=float)
X_tech_ids = df_tech[[CFG.ROW_ID_COL]].values

In [ ]:
X_trainval_id, y_trainval_combo, X_test_id, y_test_combo = iterative_train_test_split(
    X_tech_ids, y_combo, test_size=CFG.TEST_RATIO
)
val_ratio_adjusted = CFG.VAL_RATIO / (1 - CFG.TEST_RATIO)
X_train_id, y_train_combo, X_val_id, y_val_combo = iterative_train_test_split(
    X_trainval_id, y_trainval_combo, test_size=val_ratio_adjusted
)

Reconstruct DataFrames from split IDs and fit the MultiLabelBinarizer strictly on the training set to define aspect classes without data leakage.

In [ ]:
def get_df_from_ids(X_id_array):
    ids = [x[0] for x in X_id_array]
    return df_tech[df_tech[CFG.ROW_ID_COL].isin(ids)].copy()

In [ ]:
train_df_raw = get_df_from_ids(X_train_id)
val_df_raw   = get_df_from_ids(X_val_id)
test_df_raw  = get_df_from_ids(X_test_id)

mlb = MultiLabelBinarizer()
mlb.fit(train_df_raw['aspects_parsed'].tolist())

ASPECT_CLASSES = list(mlb.classes_)
N_ASPECTS      = len(ASPECT_CLASSES)

print(f'\nAspect classes ({N_ASPECTS}): {ASPECT_CLASSES}')
train_aspects_list = train_df_raw['aspects_parsed'].tolist()
for cls in ASPECT_CLASSES:
    cnt = sum(cls in a for a in train_aspects_list)
    print(f'  {cls:<20}: {cnt} train samples ({cnt/len(train_df_raw)*100:.1f}%)')


Aspect classes (7): ['battery', 'design & build', 'display & audio', 'input', 'performance', 'price', 'software']
  battery             : 54 train samples (6.7%)
  design & build      : 245 train samples (30.2%)
  display & audio     : 104 train samples (12.8%)
  input               : 72 train samples (8.9%)
  performance         : 216 train samples (26.6%)
  price               : 109 train samples (13.4%)
  software            : 111 train samples (13.7%)


Transform parsed aspects into multi-hot arrays and perform set-intersection assertions

In [ ]:
def apply_aspect_labels(df_raw):
    d      = df_raw.copy()
    y_m1   = mlb.transform(d['aspects_parsed'].tolist())
    d['labels'] = list(y_m1)
    return d

In [ ]:
train_df = apply_aspect_labels(train_df_raw)
val_df   = apply_aspect_labels(val_df_raw)
test_df  = apply_aspect_labels(test_df_raw)

y_train = np.array(train_df['labels'].tolist(), dtype=float)
y_val   = np.array(val_df['labels'].tolist(),   dtype=float)
y_test  = np.array(test_df['labels'].tolist(),  dtype=float)

train_df = train_df.drop(columns=['labels'])
val_df   = val_df.drop(columns=['labels'])
test_df  = test_df.drop(columns=['labels'])

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test (LOCKED): {len(test_df)}')
print(f'y_train shape: {y_train.shape} | y_val: {y_val.shape} | y_test: {y_test.shape}')
print('\nLabel distribution per aspect (train):')
for i, cls in enumerate(ASPECT_CLASSES):
    print(f'  {cls:<20}: {int(y_train[:,i].sum())} positives')

Train: 812 | Val: 175 | Test (LOCKED): 180
y_train shape: (812, 7) | y_val: (175, 7) | y_test: (180, 7)

Label distribution per aspect (train):
  battery             : 54 positives
  design & build      : 245 positives
  display & audio     : 104 positives
  input               : 72 positives
  performance         : 216 positives
  price               : 109 positives
  software            : 111 positives


In [ ]:
train_ids_set = set(train_df[CFG.ROW_ID_COL])
val_ids_set   = set(val_df[CFG.ROW_ID_COL])
test_ids_set  = set(test_df[CFG.ROW_ID_COL])

# Protection
assert len(train_ids_set & val_ids_set) == 0,  "LEAK: train and val NOT EMPTY!"
assert len(train_ids_set & test_ids_set) == 0, "LEAK: train and test NOT EMPTY!"
assert len(val_ids_set & test_ids_set) == 0,   "LEAK: val and test NOT EMPTY!"
print("Split integrity check: OK! NO LEAK.")

df_full_train = train_df.copy()
df_full_val   = val_df.copy()
df_full_test  = test_df.copy()

Split integrity check: OK! NO LEAK.


Stratify-split non-technical rows, concatenate them with technical sets, and assert mutual exclusivity

In [ ]:
df_non_tech = df[
    ~df[CFG.ROW_ID_COL].isin(df_tech[CFG.ROW_ID_COL])
].copy()

non_tech_train, temp = train_test_split(df_non_tech, test_size=CFG.TEST_RATIO + CFG.VAL_RATIO,
                                         stratify=df_non_tech[CFG.IS_RELATED_COL], random_state=CFG.SEED)
non_tech_val, non_tech_test = train_test_split(temp, test_size=0.5,
                                                stratify=temp[CFG.IS_RELATED_COL], random_state=CFG.SEED)

df_full_train = pd.concat([df_full_train, non_tech_train]).reset_index(drop=True)
df_full_val   = pd.concat([df_full_val,   non_tech_val]).reset_index(drop=True)
df_full_test  = pd.concat([df_full_test,  non_tech_test]).reset_index(drop=True)

In [ ]:
full_train_ids = set(df_full_train[CFG.ROW_ID_COL])
full_val_ids   = set(df_full_val[CFG.ROW_ID_COL])
full_test_ids  = set(df_full_test[CFG.ROW_ID_COL])

assert len(full_train_ids & full_val_ids)  == 0, "LEAK: full train ∩ val NOT EMPTY!"
assert len(full_train_ids & full_test_ids) == 0, "LEAK: full train ∩ test NOT EMPTY!"
assert len(full_val_ids   & full_test_ids) == 0, "LEAK: full val ∩ test NOT EMPTY!"
print("Split integrity check (full): OK! NO LEAK.")

print(f'\nFull splits (all rows):')
print(f'  Train: {len(df_full_train)} | Val: {len(df_full_val)} | Test: {len(df_full_test)}')

Split integrity check (full): OK! NO LEAK.

Full splits (all rows):
  Train: 1682 | Val: 361 | Test: 367


## 4. Data Augmentation (Back-translation for rare aspect classes)

Augment TRAIN SET, rare class (< threshold) using Helsinki-NLP. Threshold is default 150 samples because the non-rare classes have at least ~250 samples, so 150 is a reasonable cutoff to boost the tail without over-augmenting.

Get rare classes based on training set distribution

In [ ]:
def get_rare_classes(y_train, threshold=CFG.AUG_RARE_THRESHOLD):
    rare = []
    for i, cls in enumerate(ASPECT_CLASSES):
        count = int(y_train[:, i].sum())
        if count < threshold:
            rare.append((i, cls, count))
    return rare

rare_classes = get_rare_classes(y_train)
print(f'Rare classes (< {CFG.AUG_RARE_THRESHOLD} samples):')
for idx, cls, cnt in rare_classes:
    print(f'  [{idx}] {cls}: {cnt} samples')

Rare classes (< 150 samples):
  [0] battery: 54 samples
  [2] display & audio: 104 samples
  [3] input: 72 samples
  [5] price: 109 samples
  [6] software: 111 samples


Initialize data augmentation pipelines using either MarianMT-based Back-Translation (English-German-English) to enrich the training data.

**NOTE**: WordNet-based Synonym Replacement (EDA) is fallback if Back-Translation is too slow or resource-intensive, but it may produce less fluent augmentations. Also after testing, EDA is much worse compared to Back-Translation.

In [ ]:
USE_BACK_TRANSLATION = True  # Set False to use EDA

if USE_BACK_TRANSLATION:
    from transformers import MarianMTModel, MarianTokenizer

    def load_mt_models():
        tok_en_de = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-en-de')
        mdl_en_de = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-en-de').to(DEVICE)
        tok_de_en = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-de-en')
        mdl_de_en = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-de-en').to(DEVICE)
        return tok_en_de, mdl_en_de, tok_de_en, mdl_de_en

    def back_translate_batch(texts, tok_en_de, mdl_en_de, tok_de_en, mdl_de_en, batch_size=32):
        results = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tok_en_de(batch,
                            return_tensors='pt',
                            padding=True,
                            truncation=True,
                            max_length=128).to(DEVICE)

            with torch.no_grad():
                de_ids = mdl_en_de.generate(**enc,
                                            num_beams=4,
                                            max_length=128)

            de_texts = tok_en_de.batch_decode(de_ids, skip_special_tokens=True)

            enc2 = tok_de_en(de_texts,
                             return_tensors='pt',
                             padding=True,
                             truncation=True,
                             max_length=128).to(DEVICE)

            with torch.no_grad():
                en_ids = mdl_de_en.generate(**enc2, num_beams=4, max_length=128)
            en_texts = tok_de_en.batch_decode(en_ids, skip_special_tokens=True)
            results.extend(en_texts)
        return results

    # Check if back-translated text is valid (not too short and still in English), otherwise fallback to original text
    def validate_translations(orig_texts, trans_texts, min_words=5):
        validated = []
        fallback_count = 0
        for orig, trans in zip(orig_texts, trans_texts):
            use_orig = False
            if len(trans.split()) < min_words:
                use_orig = True
            else:
                try:
                    if detect(trans) != 'en':
                        use_orig = True
                except Exception:
                    use_orig = True  # langdetect fail →  fallback
            if use_orig:
                validated.append(orig)
                fallback_count += 1
            else:
                validated.append(trans)
        if fallback_count > 0:
            print(f'{fallback_count}/{len(orig_texts)} translations got fallback to OG :sob: (degenerate/non-EN)')
        return validated

    print('Loading MarianMT models for back-translation :catloading: ')
    tok_en_de, mdl_en_de, tok_de_en, mdl_de_en = load_mt_models()
    print('MarianMT loaded.')
else:
    aug_eda = naw.SynonymAug(aug_src='wordnet', aug_p=0.2)
    print('Using EDA (synonym augmentation) as fallback.')

Loading MarianMT models for back-translation :catloading: 


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

MarianMT loaded.


Filter training samples containing rare aspect classes, generate augmented texts using the chosen strategy, and append them with new UUIDs to balance the dataset.

**NOTE**: This approach do have a downside, where if a sample has multiple aspects and only one of them is rare, we will still augment that sample and keep all aspect labels. This may introduce some noise but is a simpler approach than trying to modify aspect labels per augmentation. Moreover, with too lack of labeled data, we have to accept some noise in exchange for more samples of the rare classes.

In [ ]:
def augment_rare_classes(train_df, y_train, rare_classes, num_aug=CFG.AUG_NUM_PER_SAMPLE):
    aug_rows = []
    to_aug_mask = np.zeros(len(train_df), dtype=bool)
    for i, (aspect_idx, cls, cnt) in enumerate(rare_classes):
        to_aug_mask |= (y_train[:, aspect_idx] == 1)

    to_aug_df = train_df[to_aug_mask].reset_index(drop=True)
    to_aug_labels = y_train[to_aug_mask]
    print(f'Augmenting {len(to_aug_df)} rare-class rows x{num_aug}...')
    texts_to_aug = to_aug_df[CFG.TEXT_COL].tolist()

    for k in range(num_aug):
        print(f'  Augmentation pass {k+1}/{num_aug}...')
        if USE_BACK_TRANSLATION:
            raw_aug = back_translate_batch(texts_to_aug, tok_en_de, mdl_en_de, tok_de_en, mdl_de_en)
            aug_texts = validate_translations(texts_to_aug, raw_aug)
        else:
            aug_texts = [aug_eda.augment(t)[0] if pd.notna(t) else t for t in texts_to_aug]

        for j, aug_text in enumerate(aug_texts):
            row = to_aug_df.iloc[j].copy()
            row[CFG.TEXT_COL] = aug_text
            row[CFG.ROW_ID_COL] = str(uuid.uuid4())
            aug_rows.append(row)

    aug_extra_df = pd.DataFrame(aug_rows).reset_index(drop=True)
    aug_extra_y = np.tile(to_aug_labels, (num_aug, 1)).astype(np.float32)
    aug_extra_df[CFG.IS_RELATED_COL] = 1
    aug_extra_df[CFG.IS_TECH_COL] = 1
    aug_extra_df['aspect_labels']    = aug_extra_y.tolist()
    aug_extra_df['has_aspect_label'] = True

    train_df_safe = train_df.copy()
    train_df_safe[CFG.IS_RELATED_COL] = 1
    train_df_safe[CFG.IS_TECH_COL] = 1
    train_df_safe['aspect_labels']    = y_train.astype(np.float32).tolist()
    train_df_safe['has_aspect_label'] = True

    train_aug_df = pd.concat([train_df_safe, aug_extra_df]).reset_index(drop=True)
    print(f'Train after aug: {len(train_aug_df)} rows (was {len(train_df)})')
    return train_aug_df

In [ ]:
if rare_classes:
    train_aug_df = augment_rare_classes(train_df, y_train, rare_classes)
else:
    print('No rare classes found, skipping augmentation.')
    train_aug_df = train_df.copy()
    train_aug_df[CFG.IS_RELATED_COL] = 1
    train_aug_df[CFG.IS_TECH_COL] = 1
    train_aug_df['aspect_labels']    = y_train.astype(np.float32).tolist()
    train_aug_df['has_aspect_label'] = True

# Non-tech rows: aspect_labels = sentinel, has_aspect_label = False
non_tech_train_safe = non_tech_train.copy()
non_tech_train_safe['aspect_labels']    = [
    np.full(N_ASPECTS, -1.0, dtype=np.float32).tolist()
    for _ in range(len(non_tech_train_safe))
]
non_tech_train_safe['has_aspect_label'] = False

df_full_train = pd.concat([train_aug_df, non_tech_train_safe]).sample(frac=1, random_state=CFG.SEED).reset_index(drop=True)
print(f'df_full_train: {len(df_full_train)} rows | technical = {df_full_train["has_aspect_label"].sum()}')

Augmenting 435 rare-class rows x2...
  Augmentation pass 1/2...
42/435 translations got fallback to OG :sob: (degenerate/non-EN)
  Augmentation pass 2/2...
42/435 translations got fallback to OG :sob: (degenerate/non-EN)
Train after aug: 1682 rows (was 812)
df_full_train: 2552 rows | technical = 1682


## 5. DAPT — Domain-Adaptive Pretraining (MLM)
Perform Domain-Adaptive Pretraining (DAPT) via Masked Language Modeling (MLM).

**NOTE**:
- We combine the training sentences with a filtered subset of the unlabeled pool (after removing any sentences that appear in val/test to prevent leakage)

In [ ]:
print('Loading unlabeled pool...')
df_unlabeled = pd.read_csv(CFG.UNLABELED_PATH)
print(f'Unlabeled pool: {len(df_unlabeled)} rows')
print('Columns:', df_unlabeled.columns.tolist())

# Filter out validation and test
UNLABELED_TEXT_COL = CFG.TEXT_COL
val_test_texts = set(
    df_full_val[CFG.TEXT_COL].astype(str).tolist() +
    df_full_test[CFG.TEXT_COL].astype(str).tolist()
)

df_unlabeled = df_unlabeled[~df_unlabeled[UNLABELED_TEXT_COL].astype(str).isin(val_test_texts)]
unlabeled_texts = df_unlabeled[UNLABELED_TEXT_COL].dropna().astype(str).tolist()
print(f'Valid unlabeled sentences (after filtering Val/Test leaks): {len(unlabeled_texts)}')

if len(unlabeled_texts) > CFG.DAPT_UNLABELED_SAMPLE:
    np.random.shuffle(unlabeled_texts)
    unlabeled_texts = unlabeled_texts[:CFG.DAPT_UNLABELED_SAMPLE]
    print(f'Sampled {CFG.DAPT_UNLABELED_SAMPLE} sentences for DAPT')

# Merge both unlabeled (300k rows) with labeled sentences (augmented train set) for DAPT
train_texts_for_dapt = df_full_train[CFG.TEXT_COL].astype(str).tolist()
dapt_texts = train_texts_for_dapt + unlabeled_texts
print(f'Total DAPT texts: {len(dapt_texts)}')

Loading unlabeled pool...
Unlabeled pool: 1485415 rows
Columns: ['sentence_for_model']
Valid unlabeled sentences (after filtering Val/Test leaks): 1485341
Sampled 300000 sentences for DAPT
Total DAPT texts: 302552


In [ ]:
def run_dapt(texts, backbone=CFG.BACKBONE, output_dir=CFG.DAPT_OUTPUT,
             max_len=CFG.DAPT_MAX_LEN, batch_size=CFG.DAPT_BATCH,
             epochs=CFG.DAPT_EPOCHS, lr=CFG.DAPT_LR):

    # Guard: Check if DAPT checkpoint already exists to prevent redundant training
    if os.path.exists(output_dir + '/config.json'):
        print(f'DAPT checkpoint found at {output_dir}, skipping DAPT.')
        return

    tokenizer = AutoTokenizer.from_pretrained(backbone)
    model = AutoModelForMaskedLM.from_pretrained(backbone).to(DEVICE)

    # Change data into HF Dataset for memory-efficient tokenization and training
    print(f'Tokenizing {len(texts)} sentences using HF Datasets (Memory Efficient)...')
    raw_dataset = HFDataset.from_dict({"text": texts})

    def tokenize_function(examples):
        return tokenizer(
            examples["text"], truncation=True, padding='max_length',
            max_length=max_len, return_special_tokens_mask=True
        )

    tokenized_dataset = raw_dataset.map(tokenize_function, batched=True, num_proc=2, remove_columns=["text"])
    tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "special_tokens_mask"])

    # Configure the DataCollator to randomly mask 15% of tokens
    collator   = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
    loader     = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=True,
                            collate_fn=collator, num_workers=2)

    optimizer  = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler  = get_linear_schedule_with_warmup(optimizer, int(total_steps*0.06), total_steps)

    model.train()
    checkpoint_dir = output_dir + "_checkpoints"
    os.makedirs(checkpoint_dir, exist_ok=True)

    for epoch in range(epochs):
        total_loss = 0
        for step, batch in enumerate(loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

            # show log for easier tracking (DAPT is extremely long)
            if step % 500 == 0:
                print(f'  Epoch {epoch+1}/{epochs} | Step {step}/{len(loader)} | Loss {loss.item():.4f}')

            # Auto-save checkpoint
            if step > 0 and step % 5000 == 0:
                print(f"Auto-saving intermediate checkpoint at step {step}...")
                model.save_pretrained(checkpoint_dir)
                tokenizer.save_pretrained(checkpoint_dir)

        print(f'Epoch {epoch+1} avg loss: {total_loss/len(loader):.4f}')

    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f'DAPT model saved to {output_dir}')

run_dapt(dapt_texts)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Tokenizing 302552 sentences using HF Datasets (Memory Efficient)...


Map (num_proc=2):   0%|          | 0/302552 [00:00<?, ? examples/s]

  Epoch 1/4 | Step 0/9455 | Loss 2.0276
  Epoch 1/4 | Step 500/9455 | Loss 1.3777
  Epoch 1/4 | Step 1000/9455 | Loss 1.8668
  Epoch 1/4 | Step 1500/9455 | Loss 2.6449
  Epoch 1/4 | Step 2000/9455 | Loss 2.2815
  Epoch 1/4 | Step 2500/9455 | Loss 2.0162
  Epoch 1/4 | Step 3000/9455 | Loss 2.4026
  Epoch 1/4 | Step 3500/9455 | Loss 1.8467
  Epoch 1/4 | Step 4000/9455 | Loss 2.3630
  Epoch 1/4 | Step 4500/9455 | Loss 1.7724
  Epoch 1/4 | Step 5000/9455 | Loss 2.4510
Auto-saving intermediate checkpoint at step 5000...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  Epoch 3/4 | Step 5500/9455 | Loss 1.9302
  Epoch 3/4 | Step 6000/9455 | Loss 1.6156
  Epoch 3/4 | Step 6500/9455 | Loss 1.4683
  Epoch 3/4 | Step 7000/9455 | Loss 1.9476
  Epoch 3/4 | Step 7500/9455 | Loss 1.5071
  Epoch 3/4 | Step 8000/9455 | Loss 2.0407
  Epoch 3/4 | Step 8500/9455 | Loss 1.5705
  Epoch 3/4 | Step 9000/9455 | Loss 1.5487
Epoch 3 avg loss: 1.7337
  Epoch 4/4 | Step 0/9455 | Loss 1.7389
  Epoch 4/4 | Step 500/9455 | Loss 1.5572
  Epoch 4/4 | Step 1000/9455 | Loss 1.5479
  Epoch 4/4 | Step 1500/9455 | Loss 1.0965
  Epoch 4/4 | Step 2000/9455 | Loss 1.5965
  Epoch 4/4 | Step 2500/9455 | Loss 1.1510
  Epoch 4/4 | Step 3000/9455 | Loss 1.3021
  Epoch 4/4 | Step 3500/9455 | Loss 1.4669
  Epoch 4/4 | Step 4000/9455 | Loss 1.5585
  Epoch 4/4 | Step 4500/9455 | Loss 1.7172
  Epoch 4/4 | Step 5000/9455 | Loss 1.6886
Auto-saving intermediate checkpoint at step 5000...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Epoch 4/4 | Step 5500/9455 | Loss 1.4744
  Epoch 4/4 | Step 6000/9455 | Loss 2.0285
  Epoch 4/4 | Step 6500/9455 | Loss 2.0160
  Epoch 4/4 | Step 7000/9455 | Loss 1.1185
  Epoch 4/4 | Step 7500/9455 | Loss 1.4401
  Epoch 4/4 | Step 8000/9455 | Loss 1.5346
  Epoch 4/4 | Step 8500/9455 | Loss 1.4539
  Epoch 4/4 | Step 9000/9455 | Loss 2.1528
Epoch 4 avg loss: 1.6349


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DAPT model saved to ./dapt_roberta


TESTING

In [ ]:
from transformers import pipeline as hf_pipeline

base_tokenizer = AutoTokenizer.from_pretrained(CFG.BACKBONE)
mask_token = base_tokenizer.mask_token

base_fill_mask = hf_pipeline("fill-mask", model=CFG.BACKBONE, tokenizer=CFG.BACKBONE, device=0)
dapt_fill_mask = hf_pipeline("fill-mask", model=CFG.DAPT_OUTPUT, tokenizer=CFG.DAPT_OUTPUT, device=0)

test_sentences = [
    f"The phone is amazing, but the {mask_token} life is absolutely terrible.",
    f"I love the design, but the {mask_token} quality feels very cheap and plasticky.",
    f"The laptop is super fast, but the cooling {mask_token} is too loud when gaming.",
    f"The hardware is solid, but the {mask_token} is very buggy and crashes often.",
    f"The video quality is crystal clear, however the {mask_token} is muffled and low.",
]

for i, text in enumerate(test_sentences):
    print(f"\n[{i+1}] Test: {text}")
    base_preds = [res['token_str'].strip() for res in base_fill_mask(text)[:3]]
    print(f" ➔ Base Model: {base_preds}")
    dapt_preds = [res['token_str'].strip() for res in dapt_fill_mask(text)[:3]]
    print(f" ➔ DAPT Model: {dapt_preds}")
    print("-" * 50)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]


[1] Test: The phone is amazing, but the <mask> life is absolutely terrible.
 ➔ Base Model: ['battery', 'Battery', 'sex']
 ➔ DAPT Model: ['battery', 'batter', 'charger']
--------------------------------------------------

[2] Test: I love the design, but the <mask> quality feels very cheap and plasticky.
 ➔ Base Model: ['build', 'production', 'overall']
 ➔ DAPT Model: ['build', 'material', 'built']
--------------------------------------------------

[3] Test: The laptop is super fast, but the cooling <mask> is too loud when gaming.
 ➔ Base Model: ['fan', 'noise', 'system']
 ➔ DAPT Model: ['fan', 'system', 'pad']
--------------------------------------------------

[4] Test: The hardware is solid, but the <mask> is very buggy and crashes often.
 ➔ Base Model: ['software', 'UI', 'OS']
 ➔ DAPT Model: ['software', 'system', 'computer']
--------------------------------------------------

[5] Test: The video quality is crystal clear, however the <mask> is muffled and low.
 ➔ Base Model: ['aud

## 6. Model 1 — Multi-task Aspect Detection
Approach: Multi-task fine-tuning of a Domain-Adapted RoBERTa + LoRA for 3 classification heads.

Example: sentence → Head 1: Related/Unrelated → Head 2: General/Technical → Head 3: Aspect (multi-label N-class).

Dataset construction:
Technical rows (is_related=1, is_technical=1) are split using iterative_train_test_split (multilabel-safe) on combo labels (aspect_sentiment pairs) to preserve multi-label distribution. Non-technical rows are appended to each split using a separate stratified split on is_related. Labels are assigned to the full splits via UUID-based row_id join. Back-translation augmentation (EN→DE→EN via Helsinki-NLP MarianMT) is applied on the training set for rare aspect classes (< threshold); augmented rows carry the original ground-truth aspect labels directly.

Fine-tuning strategy:

Base: Domain-Adapted RoBERTa + LoRA (r=32, alpha=32, target=[query, key, value, dense]) — only LoRA matrices + 3 heads are trainable
Head 1 (Related/Unrelated): BCE on all samples, loss weight 0.2
Head 2 (General/Technical): BCE masked to rows where label_related == 1 (ground-truth, not Head 1 prediction), loss weight 0.2
Head 3 (Aspect, multi-label): BCE with pos_weight masked to rows where label_related == 1 AND label_technical == 1 AND has_aspect_label == True (all ground-truth), loss weight 0.6
Discriminative LR: LoRA params lr, classifier heads lr × 5
Linear warmup (10% of total steps) + Head 1 frozen after epoch 10
Early stopping monitors configurable Val metric (default: mean F1 across all 3 heads)

Threshold tuning: After training, sweep on validation set per-head —
Head 1 & 2: F-beta (β=2) sweep → auto-select THRESHOLD_H1, THRESHOLD_H2
Head 3: Per-class F1 sweep → auto-select per-class THRESHOLDS_H3

**NOTE**: We currently cannot justify using prediction of one head for another head due to the lack of labeled data. This will drag the final performance down badly. Moreover, this approach is much better, at least in this case, because after finishing both model, our future plan is to making more psuedo-labeled data using the unlabeled data previous. It will be labeled with is_related, is_technical, and with aspects.

Data Alignment & Label Assignment via UUID

In [ ]:
def assign_aspect_labels_by_id(df_split, ref_df, y_aspects, n_aspects):
    assert len(ref_df) == len(y_aspects), (
        f"MISALIGNMENT: ref_df có {len(ref_df)} rows but y_aspects has {len(y_aspects)} rows! "
        "Ensure ref_df is output directly from make_df_from_split, hasnt modified."
    )
    # Guard: row_id in ref_df has to be unique
    assert ref_df[CFG.ROW_ID_COL].is_unique, "row_id in ref_df not unique — data is corrupted!"

    # Build ID → label dict by positional index
    id_to_label = {
        str(ref_df[CFG.ROW_ID_COL].iloc[i]): y_aspects[i].astype(np.float32).tolist()
        for i in range(len(ref_df))
    }

    df_out = df_split.copy().reset_index(drop=True)

    def lookup(rid):
        rid = str(rid)
        if rid in id_to_label:
            return id_to_label[rid], True
        return np.full(n_aspects, -1.0, dtype=np.float32).tolist(), False

    mapped = df_out[CFG.ROW_ID_COL].map(lookup)
    df_out['aspect_labels']    = [m[0] for m in mapped]
    df_out['has_aspect_label'] = [m[1] for m in mapped] 

    matched = df_out['has_aspect_label'].sum()
    print(f'  → {matched}/{len(df_out)} rows matched by row_id (Expected = {len(ref_df)})')
    if matched != len(ref_df):
        print(f'ERRORRROO  {len(ref_df) - matched} rows in ref_df cannot be found in df_split!')

    return df_out

In [ ]:
# aspect_labels ->  val and test using row_id-based join
df_full_val  = assign_aspect_labels_by_id(df_full_val,  val_df,  y_val,  N_ASPECTS)
df_full_test = assign_aspect_labels_by_id(df_full_test, test_df, y_test, N_ASPECTS)

# df_full_train has 'aspect_labels' and 'has_aspect_label' from 'augmentation arc' -> Verify df_full_train has 'has_aspect_label'
assert 'has_aspect_label' in df_full_train.columns, "df_full_train missing has_aspect_label!"

print('\nSummary:')
for name, df in [('df_full_train', df_full_train),
                 ('df_full_val',   df_full_val),
                 ('df_full_test',  df_full_test)]:
    n_tech = df['has_aspect_label'].sum()
    print(f'{name:<20}: {len(df)} rows | has_aspect_label=True: {n_tech}')

  → 175/361 rows matched by row_id (Expected = 175)
  → 180/367 rows matched by row_id (Expected = 180)

Summary:
df_full_train       : 2552 rows | has_aspect_label=True: 1682
df_full_val         : 361 rows | has_aspect_label=True: 175
df_full_test        : 367 rows | has_aspect_label=True: 180


PyTorch Multi-Task Dataset Initialization

In [ ]:
class MultiTaskDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=CFG.M1_MAX_LEN):
        assert 'aspect_labels' in df.columns,    "Missing 'aspect_labels'!"
        assert 'has_aspect_label' in df.columns, "Missing 'has_aspect_label'!"

        self.texts          = df[CFG.TEXT_COL].astype(str).tolist()
        self.is_related     = df[CFG.IS_RELATED_COL].fillna(0).astype(int).tolist()
        self.is_technical   = df[CFG.IS_TECH_COL].fillna(0).astype(int).tolist()
        self.has_asp_label  = df['has_aspect_label'].astype(bool).tolist() 
        self.y_aspects      = [
            np.array(lbl, dtype=np.float32)
            for lbl in df['aspect_labels'].tolist()
        ]
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors='pt',
        )
        return {
            'input_ids':         enc['input_ids'].squeeze(0),
            'attention_mask':    enc['attention_mask'].squeeze(0),
            'label_related':     torch.tensor(self.is_related[idx],   dtype=torch.float32),
            'label_technical':   torch.tensor(self.is_technical[idx], dtype=torch.float32),
            'label_aspects':     torch.tensor(self.y_aspects[idx],    dtype=torch.float32),
            'has_aspect_label':  torch.tensor(self.has_asp_label[idx], dtype=torch.bool), 
        }

In [ ]:
BACKBONE_FOR_M1 = (
    CFG.DAPT_OUTPUT if os.path.exists(CFG.DAPT_OUTPUT + '/config.json')
    else CFG.BACKBONE
)
print(f'Using backbone: {BACKBONE_FOR_M1}')
m1_tokenizer = AutoTokenizer.from_pretrained(BACKBONE_FOR_M1)

ds_train = MultiTaskDataset(df_full_train, m1_tokenizer)
ds_val   = MultiTaskDataset(df_full_val,   m1_tokenizer)
ds_test  = MultiTaskDataset(df_full_test,  m1_tokenizer)

print(f'Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}')

s = ds_train[0]
assert s['label_aspects'].shape[0] == N_ASPECTS
print(f'  label_related={s["label_related"].item()} '
      f'label_technical={s["label_technical"].item()} '
      f'has_aspect_label={s["has_aspect_label"].item()} '
      f'label_aspects[:3]={s["label_aspects"][:3].tolist()} ✓')

Using backbone: ./dapt_roberta
Train: 2552 | Val: 361 | Test: 367
  label_related=1.0 label_technical=1.0 has_aspect_label=True label_aspects[:3]=[0.0, 1.0, 0.0] ✓


Class Imbalance Handling for BCE Loss

In [ ]:
def compute_pos_weight(df_train, n_aspects):
    # pos_weight[i] = n_neg[i] / n_pos[i] ~> BCE class-rebalancing.

    # Filter boolean flag
    tech_mask = df_train['has_aspect_label'].values.astype(bool)
    tech_labels = np.array(
        [row for row, flag in zip(df_train['aspect_labels'].tolist(), tech_mask) if flag],
        dtype=np.float32
    )
    assert len(tech_labels) > 0, "NO technical rows in df_full_train!!!"

    n_pos = tech_labels.sum(axis=0)
    n_neg = len(tech_labels) - n_pos
    pos_w = np.clip(n_neg / (n_pos + 1e-6), 0.1, 10.0)

    print(f'pos_weight ({len(tech_labels)} technical rows):')
    for i, cls in enumerate(ASPECT_CLASSES):
        print(f'  {cls:<20}: {pos_w[i]:.3f}')
    return torch.tensor(pos_w, dtype=torch.float32)


pos_weight_h3 = compute_pos_weight(df_full_train, N_ASPECTS)

pos_weight (1682 technical rows):
  battery             : 9.383
  design & build      : 5.161
  display & audio     : 4.391
  input               : 6.787
  performance         : 4.760
  price               : 4.144
  software            : 4.051


Multi-Task Architecture & Conditional Loss Strategy

Structure: RoBERTa Backbone (LoRA) + 3 HEADS
All 3 heads share the same [CLS] token embedding from the encoder.

- Head 1: Related/Unrelated  → BCE on ALL samples
- Head 2: General/Technical  → BCE masked to rows where label_related == 1
- Head 3: Aspect (multi-label N-class) → BCE (+ pos_weight) masked to rows where
                                          label_related == 1 AND label_technical == 1
                                          AND has_aspect_label == True

Loss aggregation (default):
  total_loss = 0.2 * loss_h1 + 0.2 * loss_h2 + 0.6 * loss_h3

Alternative (USE_UNCERTAINTY_WEIGHTS = True):
  Homoscedastic uncertainty weighting — each head has a learnable log_var parameter.
  loss = Σ [ exp(-log_var_i) * loss_i + log_var_i ]
  log_var is clamped to [-4.0, 4.0] for stability.
  If a head has no valid samples in the batch, its term is skipped entirely.

In [ ]:
class MultiTaskAspectModel(nn.Module):
    """
    Trainable: LoRA matrices (~1.5M params) + 3 heads.
    Frozen   : The rest of RoBERTa (~123M params).
    """
    def __init__(self, backbone_name, n_aspects,
                 use_uncertainty=CFG.USE_UNCERTAINTY_WEIGHTS):
        super().__init__()

        base_encoder = AutoModel.from_pretrained(backbone_name)
        hidden = base_encoder.config.hidden_size

        lora_cfg = LoraConfig(
            r=CFG.M1_LORA_R,
            lora_alpha=CFG.M1_LORA_ALPHA,
            target_modules=CFG.M1_LORA_TARGET,
            lora_dropout=CFG.M1_LORA_DROPOUT,
            bias='none',
        )
        self.encoder = get_peft_model(base_encoder, lora_cfg)
        self.encoder.print_trainable_parameters()

        self.head1 = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, 1))
        self.head2 = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, 1))
        self.head3 = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, n_aspects))

        self.use_uncertainty = use_uncertainty
        if use_uncertainty:
            self.log_var_h1 = nn.Parameter(torch.zeros(1))
            self.log_var_h2 = nn.Parameter(torch.zeros(1))
            self.log_var_h3 = nn.Parameter(torch.zeros(1))

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return {
            'h1': self.head1(cls).squeeze(-1),
            'h2': self.head2(cls).squeeze(-1),
            'h3': self.head3(cls),
        }

    def compute_loss(self, logits, batch, pos_weight_h3=None):
        h1_logits = logits['h1']
        h2_logits = logits['h2']
        h3_logits = logits['h3']

        lbl_rel     = batch['label_related']
        lbl_tech    = batch['label_technical']
        lbl_asp     = batch['label_aspects']
        has_asp_lbl = batch['has_aspect_label']

        # Head 1 — all samples
        loss_h1 = F.binary_cross_entropy_with_logits(h1_logits, lbl_rel)

        # Head 2 — related only
        related_mask = (lbl_rel == 1)
        h2_has_data  = related_mask.any()
        if h2_has_data:
            loss_h2 = F.binary_cross_entropy_with_logits(
                h2_logits[related_mask], lbl_tech[related_mask]
            )
        else:
            loss_h2 = (h2_logits * 0).mean()

        # Head 3 — technical rows with aspect label
        tech_mask  = (lbl_rel == 1) & (lbl_tech == 1)
        final_mask = tech_mask & has_asp_lbl
        h3_has_data = final_mask.any()
        if h3_has_data:
            h3_sel = h3_logits[final_mask]
            y3_sel = lbl_asp[final_mask]
            if pos_weight_h3 is not None:
                loss_h3 = F.binary_cross_entropy_with_logits(
                    h3_sel, y3_sel,
                    pos_weight=pos_weight_h3.to(h3_sel.device)
                )
            else:
                loss_h3 = F.binary_cross_entropy_with_logits(h3_sel, y3_sel)
        else:
            loss_h3 = (h3_logits * 0).mean()

        if self.use_uncertainty:
            lv1 = self.log_var_h1.clamp(-4.0, 4.0)
            lv2 = self.log_var_h2.clamp(-4.0, 4.0)
            lv3 = self.log_var_h3.clamp(-4.0, 4.0)
            loss = torch.exp(-lv1) * loss_h1 + lv1
            if h2_has_data:
                loss = loss + torch.exp(-lv2) * loss_h2 + lv2
            if h3_has_data:
                loss = loss + torch.exp(-lv3) * loss_h3 + lv3
        else:
            loss = (CFG.LOSS_W_H1 * loss_h1 +
                    CFG.LOSS_W_H2 * loss_h2 +
                    CFG.LOSS_W_H3 * loss_h3)

        return loss, {
            'h1': loss_h1.item(),
            'h2': loss_h2.item() if h2_has_data else 0.0,
            'h3': loss_h3.item() if h3_has_data else 0.0,
        }

print('MultiTaskAspectModel (LoRA) defined')

MultiTaskAspectModel (LoRA v5) defined


Training and evaluating

In [ ]:
def evaluate_m1(model, loader, pos_weight_h3):
    model.eval()
    all_h1_prob, all_h1_lbl = [], []
    all_h2_prob, all_h2_lbl = [], []
    all_h3_prob, all_h3_lbl = [], []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            batch   = {k: v.to(DEVICE) for k, v in batch.items()}
            logits  = model(batch['input_ids'], batch['attention_mask'])
            loss, _ = model.compute_loss(logits, batch, pos_weight_h3)
            total_loss += loss.item()

            rel     = batch['label_related'].cpu().numpy()
            tec     = batch['label_technical'].cpu().numpy()
            asp     = batch['label_aspects'].cpu().numpy()
            has_lbl = batch['has_aspect_label'].cpu().numpy()

            all_h1_prob.extend(torch.sigmoid(logits['h1']).cpu().numpy())
            all_h1_lbl.extend(rel)

            rel_mask = rel == 1
            if rel_mask.sum() > 0:
                all_h2_prob.extend(torch.sigmoid(logits['h2']).cpu().numpy()[rel_mask])
                all_h2_lbl.extend(tec[rel_mask])

            tech_mask = (rel == 1) & (tec == 1) & has_lbl
            if tech_mask.sum() > 0:
                all_h3_prob.extend(torch.sigmoid(logits['h3']).cpu().numpy()[tech_mask])
                all_h3_lbl.extend(asp[tech_mask])

    f1_h1 = f1_score(np.array(all_h1_lbl),
                     (np.array(all_h1_prob) > 0.5).astype(int),
                     average='macro', zero_division=0) if all_h1_lbl else 0.0
    f1_h2 = f1_score(np.array(all_h2_lbl),
                     (np.array(all_h2_prob) > 0.5).astype(int),
                     average='macro', zero_division=0) if all_h2_lbl else 0.0
    f1_h3 = f1_score(np.array(all_h3_lbl),
                     (np.array(all_h3_prob) > 0.5).astype(int),
                     average='macro', zero_division=0) if all_h3_lbl else 0.0

    return {
        'val_loss': total_loss / len(loader),
        'f1_h1': f1_h1, 'f1_h2': f1_h2, 'f1_h3': f1_h3,
        'f1_mean': (f1_h1 + f1_h2 + f1_h3) / 3,
    }

In [ ]:
def train_model1(train_ds, val_ds, pos_weight_h3,
                 backbone=BACKBONE_FOR_M1,
                 output_dir=CFG.M1_OUTPUT,
                 epochs=CFG.M1_EPOCHS):

    if os.path.exists(output_dir + '/model_best.pt'):
        print(f'Checkpoint found at {output_dir}, skipping training.')
        return

    train_loader = DataLoader(train_ds, batch_size=CFG.M1_BATCH, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.M1_BATCH, shuffle=False, num_workers=2)

    model = MultiTaskAspectModel(backbone, N_ASPECTS).to(DEVICE)

    lora_params = [p for p in model.encoder.parameters() if p.requires_grad]
    head_params = (list(model.head1.parameters()) +
                   list(model.head2.parameters()) +
                   list(model.head3.parameters()))
    n_lora = sum(p.numel() for p in lora_params)
    n_head = sum(p.numel() for p in head_params)
    print(f'[v5 LoRA] Trainable: LoRA={n_lora:,} | Heads={n_head:,} | Total={n_lora+n_head:,}')

    optimizer = AdamW([
        {'params': lora_params, 'lr': CFG.M1_LR},
        {'params': head_params, 'lr': CFG.M1_LR * 5},
    ], weight_decay=CFG.M1_WEIGHT_DECAY)

    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * CFG.M1_WARMUP_RATIO), total_steps
    )

    best_val_score  = 0.0
    best_h3_at_best = 0.0
    patience_ctr    = 0
    h1_frozen       = False
    os.makedirs(output_dir, exist_ok=True)

    print(f'Early stopping monitors: {CFG.M1_MONITOR_METRIC}')

    for epoch in range(epochs):
        if epoch == CFG.FREEZE_H1_AFTER_EPOCH and not h1_frozen:
            for p in model.head1.parameters():
                p.requires_grad = False
            h1_frozen = True
            print(f'  [Epoch {epoch+1}] Head 1 frozen.')

        model.train()
        epoch_loss     = 0.0
        loss_breakdown = {'h1': 0.0, 'h2': 0.0, 'h3': 0.0}

        for batch in train_loader:
            batch    = {k: v.to(DEVICE) for k, v in batch.items()}
            logits   = model(batch['input_ids'], batch['attention_mask'])
            loss, bd = model.compute_loss(logits, batch, pos_weight_h3)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            epoch_loss += loss.item()
            for k in bd:
                loss_breakdown[k] += bd[k]

        n = len(train_loader)
        print(f'Epoch {epoch+1}/{epochs} | Loss={epoch_loss/n:.4f} '
              f'[H1={loss_breakdown["h1"]/n:.3f} '
              f'H2={loss_breakdown["h2"]/n:.3f} '
              f'H3={loss_breakdown["h3"]/n:.3f}]')

        metrics = evaluate_m1(model, val_loader, pos_weight_h3)
        print(f'  Val: loss={metrics["val_loss"]:.4f} '
              f'F1_H1={metrics["f1_h1"]:.4f} '
              f'F1_H2={metrics["f1_h2"]:.4f} '
              f'F1_H3={metrics["f1_h3"]:.4f} '
              f'mean={metrics["f1_mean"]:.4f}  '
              f'[monitor={CFG.M1_MONITOR_METRIC}]')

        current_score = metrics[CFG.M1_MONITOR_METRIC]
        if current_score > best_val_score:
            best_val_score  = current_score
            best_h3_at_best = metrics['f1_h3']
            torch.save(model.state_dict(), output_dir + '/model_best.pt')
            patience_ctr = 0
            print(f'  ✓ New best {CFG.M1_MONITOR_METRIC}={best_val_score:.4f} '
                  f'(H3 F1={best_h3_at_best:.4f}) — saved.')
        else:
            patience_ctr += 1
            if patience_ctr >= CFG.M1_PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    with open(output_dir + '/config.json', 'w') as f:
        json.dump({'backbone': backbone, 'n_aspects': N_ASPECTS,
                   'aspect_classes': ASPECT_CLASSES,
                   'best_val_score': best_val_score,
                   'best_h3_f1': best_h3_at_best,
                   'monitor_metric': CFG.M1_MONITOR_METRIC,
                   'lora_r': CFG.M1_LORA_R,
                   'lora_alpha': CFG.M1_LORA_ALPHA,
                   'lora_target': CFG.M1_LORA_TARGET}, f)
    print(f'Training done. Best Val {CFG.M1_MONITOR_METRIC}: {best_val_score:.4f} | H3 F1: {best_h3_at_best:.4f}')


train_model1(ds_train, ds_val, pos_weight_h3)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: ./dapt_roberta
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 5,357,568 || all params: 130,003,200 || trainable%: 4.1211
[v5 LoRA] Trainable: LoRA=5,357,568 | Heads=6,921 | Total=5,364,489
Early stopping monitors: f1_mean
Epoch 1/50 | Loss=0.9637 [H1=0.638 H2=0.650 H3=1.177]
  Val: loss=0.6132 F1_H1=0.3808 F1_H2=0.4450 F1_H3=0.1491 mean=0.3250  [monitor=f1_mean]
  ✓ New best f1_mean=0.3250 (H3 F1=0.1491) — saved.
Epoch 2/50 | Loss=0.8717 [H1=0.594 H2=0.441 H3=1.108]
  Val: loss=0.6425 F1_H1=0.3808 F1_H2=0.4450 F1_H3=0.6215 mean=0.4824  [monitor=f1_mean]
  ✓ New best f1_mean=0.4824 (H3 F1=0.6215) — saved.
Epoch 3/50 | Loss=0.4699 [H1=0.498 H2=0.253 H3=0.533]
  Val: loss=0.3464 F1_H1=0.5974 F1_H2=0.8072 F1_H3=0.7901 mean=0.7316  [monitor=f1_mean]
  ✓ New best f1_mean=0.7316 (H3 F1=0.7901) — saved.
Epoch 4/50 | Loss=0.3065 [H1=0.422 H2=0.126 H3=0.328]
  Val: loss=0.3345 F1_H1=0.7273 F1_H2=0.8756 F1_H3=0.7798 mean=0.7942  [monitor=f1_mean]
  ✓ New best f1_mean=0.7942 (H3 F1=0.7798) — saved.
Epoch 5/50 | Loss=0.2477 [H1=0.382 H2=0.09

In [ ]:
m1_model = MultiTaskAspectModel(BACKBONE_FOR_M1, N_ASPECTS).to(DEVICE)
m1_model.load_state_dict(torch.load(CFG.M1_OUTPUT + '/model_best.pt', map_location=DEVICE))
m1_model.eval()
print('Model 1 (LoRA v5) loaded.')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: ./dapt_roberta
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 5,357,568 || all params: 130,003,200 || trainable%: 4.1211
Model 1 (LoRA v5) loaded.


Inference & Threshold Tuning

In [ ]:
def collect_predictions_m1(model, loader):
    model.eval()
    h3_probs, h3_labels = [], []
    h2_probs, h2_labels = [], []
    h1_probs, h1_labels = [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(batch['input_ids'], batch['attention_mask'])

            rel     = batch['label_related'].cpu().numpy()
            tec     = batch['label_technical'].cpu().numpy()
            asp     = batch['label_aspects'].cpu().numpy()
            has_lbl = batch['has_aspect_label'].cpu().numpy()

            h1_probs.extend(torch.sigmoid(logits['h1']).cpu().numpy())
            h1_labels.extend(rel)

            rel_mask = rel == 1
            if rel_mask.sum() > 0:
                h2_probs.extend(torch.sigmoid(logits['h2']).cpu().numpy()[rel_mask])
                h2_labels.extend(tec[rel_mask])

            tech_mask = (rel == 1) & (tec == 1) & has_lbl
            if tech_mask.sum() > 0:
                h3_probs.extend(torch.sigmoid(logits['h3']).cpu().numpy()[tech_mask])
                h3_labels.extend(asp[tech_mask])

    return (
        np.array(h1_probs), np.array(h1_labels),
        np.array(h2_probs), np.array(h2_labels),
        np.array(h3_probs), np.array(h3_labels)
    )

In [ ]:
def tune_thresholds_per_class(h3_probs, h3_labels, aspect_classes):
    thresholds = {}
    print('Per-class threshold tuning (Val set):')
    for i, cls in enumerate(aspect_classes):
        best_f1, best_th = 0, 0.5
        for th in np.arange(0.1, 0.92, 0.02):
            pred = (h3_probs[:, i] >= th).astype(int)
            f1   = f1_score(h3_labels[:, i], pred, zero_division=0)
            if f1 > best_f1:
                best_f1, best_th = f1, th
        thresholds[cls] = best_th
        print(f'  {cls:<20}: th={best_th:.2f} | Val F1={best_f1:.4f}')
    return thresholds

In [ ]:
val_loader = DataLoader(ds_val, batch_size=CFG.M1_BATCH, shuffle=False, num_workers=2)
h1_probs_val, h1_labels_val, h2_probs_val, h2_labels_val, h3_probs_val, h3_labels_val = collect_predictions_m1(m1_model, val_loader)

In [ ]:
best_th1, best_f2_h1 = 0.5, 0
for th in np.arange(0.1, 0.92, 0.05):
    # fbeta_score with beta=2.0
    f2 = fbeta_score(h1_labels_val, (h1_probs_val >= th).astype(int),
                     beta=2.0, average='macro', zero_division=0)
    if f2 > best_f2_h1:
        best_f2_h1, best_th1 = f2, th
THRESHOLD_H1 = best_th1
print(f'\nHead 1 (Unrelated) tuned threshold: {THRESHOLD_H1:.2f} (Val F2={best_f2_h1:.4f})')


Head 1 (Unrelated) tuned threshold: 0.65 (Val F2=0.8070)


In [ ]:
best_th2, best_f2_h2 = 0.5, 0
for th in np.arange(0.1, 0.92, 0.05):
    f2 = fbeta_score(h2_labels_val, (h2_probs_val >= th).astype(int),
                     beta=2.0, average='macro', zero_division=0)
    if f2 > best_f2_h2:
        best_f2_h2, best_th2 = f2, th
THRESHOLD_H2 = best_th2
print(f'Head 2 (General) tuned threshold: {THRESHOLD_H2:.2f} (Val F2={best_f2_h2:.4f})\n')

Head 2 (General) tuned threshold: 0.65 (Val F2=0.9599)



In [ ]:
THRESHOLDS_H3 = tune_thresholds_per_class(h3_probs_val, h3_labels_val, ASPECT_CLASSES)

all_thresholds = {'h1': THRESHOLD_H1, 'h2': THRESHOLD_H2, 'h3': THRESHOLDS_H3}
with open(CFG.M1_OUTPUT + '/thresholds.json', 'w') as f:
    json.dump(all_thresholds, f, indent=2)
print('\nAll thresholds saved successfully.')

Per-class threshold tuning (Val set):
  battery             : th=0.40 | Val F1=0.9286
  design & build      : th=0.24 | Val F1=0.8288
  display & audio     : th=0.10 | Val F1=0.9091
  input               : th=0.52 | Val F1=0.8750
  performance         : th=0.80 | Val F1=0.8506
  price               : th=0.54 | Val F1=0.9388
  software            : th=0.82 | Val F1=0.7917

All thresholds saved successfully.


## 7. Model 2 — Sentiment (Fine-tuned, yangheng/deberta-absa)
Approach: Fine-tune on per-aspect sentiment triples from final.csv

Example: target_aspect = "battery;camera" → target_sentiment = "0;1".

Dataset construction:
build_m2_triples() extracts (sentence, aspect, sentiment_label) triples from each split, correctly handling both single-aspect and multi-aspect rows. Length-mismatch rows are dropped entirely.

Fine-tuning strategy:

Base: yangheng/deberta-v3-base-absa-v1.1 while preserving the original input format [sentence] [SEP] [aspect]
CrossEntropyLoss + class weights to handle imbalance + label_smoothing=0.1 for calibration
Discriminative LR: backbone lr, classifier head lr × 5
Early stopping monitors Val F1-Macro

Confidence threshold: Calibrate on the validation set after training → automatically select CONF_THRESHOLD_M2.

**NOTE**: Due to the lack of Neutral data in labeled data, we choose to group Neutral and Positive reviews together, as it is the simpliest way as well as one of the best way to deal with this right now. Again, psuedo-labeled data can help on allow us to improve on this.

Transfer annotation multi-aspects to list (aspect, sentiment_int) pairs.

    aspect_str    : 'battery;camera;performance'
    sentiment_str : '-1;1;1'   ← aligned theo thứ tự, semicolon-separated

    Returns: [('battery', -1), ('camera', 1), ('performance', 0)]
    (Empty list if NaN or length mismatch)

In [ ]:
SENTIMENT_LABELS = ['Negative', 'Positive']  # index 0 | 1 |
SENTIMENT_MAP = {-1: 0, 0: 1, 1: 1}  # sentiment int → class index


def parse_aspect_sentiment_pairs(aspect_str, sentiment_str, exclude=EXCLUDE_ASPECTS):
    if pd.isna(aspect_str) or pd.isna(sentiment_str):
        return []

    raw_aspects    = [a.strip().lower() for a in str(aspect_str).split(';')]
    raw_sentiments = [s.strip()         for s in str(sentiment_str).split(';')]

    # Drop misaligned annotation
    if len(raw_aspects) != len(raw_sentiments):
        return []

    pairs = []
    for asp, sent in zip(raw_aspects, raw_sentiments):
        if asp in exclude or not asp:
            continue
        try:
            pairs.append((asp, int(float(sent))))
        except (ValueError, TypeError):
            continue
    return pairs

In [ ]:
def build_m2_triples(df_split, label_map=SENTIMENT_MAP):
    rows = []
    for _, row in df_split.iterrows():
        if row.get(CFG.IS_TECH_COL, 0) != 1 or not row.get('is_valid_pairs', False):
            continue

        sentence = str(row[CFG.TEXT_COL])
        for asp, sent_int in zip(row['aspects_parsed'], row['sentiments_parsed']):
            rows.append({
                'sentence': sentence,
                'aspect': asp,
                'label_idx': label_map.get(sent_int, 1),
            })

    return pd.DataFrame(rows)

In [ ]:
print('Building M2 per-aspect triples (Neutral dropped)...')
m2_train_df = build_m2_triples(df_full_train); print(f'  → Train : {len(m2_train_df)}')
m2_val_df   = build_m2_triples(df_full_val);   print(f'  → Val   : {len(m2_val_df)}')
m2_test_df  = build_m2_triples(df_full_test);  print(f'  → Test  : {len(m2_test_df)} (LOCKED)')

print('\nTrain distribution (sau khi drop Neutral):')
dist  = m2_train_df['label_idx'].value_counts().sort_index()
total = len(m2_train_df)
for idx, cnt in dist.items():
    print(f'  {SENTIMENT_LABELS[idx]:10}: {cnt:4d} ({cnt/total*100:.1f}%)')
assert 2 not in dist.index, 'OLD LABEL IN TRAIN SET'

Building M2 per-aspect triples (Neutral dropped)...
  → Train : 1915
  → Val   : 195
  → Test  : 195 (LOCKED)

Train distribution (sau khi drop Neutral):
  Negative  :  780 (40.7%)
  Positive  : 1135 (59.3%)


In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    """
    Input  : text=sentence, text_pair=aspect
    Label  : class index ∈ {0=Negative, 1=Positive}
    """
    def __init__(self, df, tokenizer, max_len=CFG.M2_MAX_LEN):
        # Seperate 2 list
        self.sentences = df['sentence'].tolist()
        self.aspects   = df['aspect'].tolist()

        self.labels    = df['label_idx'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        # Tokenizer will place [SEP] automatically
        enc = self.tokenizer(
            text=self.sentences[idx],
            text_pair=self.aspects[idx],
            truncation='only_first',
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
def compute_sentiment_class_weights(train_df):
    counts = train_df['label_idx'].value_counts().sort_index()
    total = len(train_df)
    weights = []

    for i in range(2):
        weights.append(float(np.clip(total / (2 * counts.get(i, 1)), 0.2, 5.0)))

    w = torch.tensor(weights, dtype=torch.float32)
    print(f'Sentiment class weights: Neg={w[0]:.3f} | Pos_Neu={w[1]:.3f}')
    return w

In [ ]:
m2_class_weights = compute_sentiment_class_weights(m2_train_df)

Sentiment class weights: Neg=1.228 | Pos_Neu=0.844


In [ ]:
m2_tokenizer = AutoTokenizer.from_pretrained(CFG.SENTIMENT_MODEL)
m2_model = AutoModelForSequenceClassification.from_pretrained(
    CFG.SENTIMENT_MODEL,
    num_labels=2,
    id2label={0: 'Negative', 1: 'Positive'},
    label2id={'Negative': 0, 'Positive': 1},
    ignore_mismatched_sizes=True
)
m2_model.to(DEVICE)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: yangheng/deberta-v3-base-absa-v1.1
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

Batch inference

In [ ]:
def predict_sentiment_batch_raw(sentences, aspects, tokenizer, model,
                                 batch_size=CFG.M2_INFERENCE_BATCH):
    all_probs   = []

    for i in range(0, len(sentences), batch_size):
        batch_sentences = sentences[i:i+batch_size]
        batch_aspects   = aspects[i:i+batch_size]

        enc   = tokenizer(
            text=batch_sentences,
            text_pair=batch_aspects,
            truncation='only_first',
            padding='max_length',
            max_length=CFG.M2_MAX_LEN,
            return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)

    all_probs = np.array(all_probs)
    return np.argmax(all_probs, axis=-1), all_probs

Fine-tune yangheng/deberta-absa per-aspect sentiment triples.

In [ ]:
def train_model2(train_df, val_df, tokenizer, model,
                 class_weights, output_dir=CFG.M2_OUTPUT,
                 epochs=CFG.M2_EPOCHS, lr=CFG.M2_LR, patience=CFG.M2_PATIENCE):
    checkpoint_path = output_dir + '/model_best_m2.pt'
    if os.path.exists(checkpoint_path) and not CFG.FORCE_RETRAIN_M2:
        print(f'M2 checkpoint found at {output_dir}, skipping training.')
        print('  → Set CFG.FORCE_RETRAIN_M2 = True to retrain.')
        return
    if os.path.exists(checkpoint_path) and CFG.FORCE_RETRAIN_M2:
        import shutil
        shutil.rmtree(output_dir)
        print(f'[FORCE_RETRAIN_M2] Delete old checkpoint {output_dir}. Train again...')

    train_ds = SentimentDataset(train_df, tokenizer)
    val_ds   = SentimentDataset(val_df,   tokenizer)

    train_loader = DataLoader(train_ds, batch_size=CFG.M2_BATCH, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=CFG.M2_BATCH, shuffle=False, num_workers=2)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE),
        label_smoothing=0.1,
    )

    # Discriminative LR: keeping ABSA knowledge
    try:
        backbone_params   = model.deberta.parameters()
    except AttributeError:
        backbone_params   = model.base_model.parameters()
    classifier_params = model.classifier.parameters()

    optimizer = AdamW(
        [
            {'params': backbone_params,'lr': lr},
            {'params': classifier_params, 'lr': lr * 5},
        ],
        weight_decay=0.01,
    )

    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps,
    )

    best_val_f1  = 0.0
    patience_ctr = 0
    os.makedirs(output_dir, exist_ok=True)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            epoch_loss += loss.item()

        model.eval()
        val_preds_ep, val_true_ep = [], []
        val_loss_total = 0.0

        with torch.no_grad():
            for batch in val_loader:
                input_ids      = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels         = batch['labels'].to(DEVICE)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss    = criterion(outputs.logits, labels)
                val_loss_total += loss.item()

                preds = outputs.logits.argmax(dim=-1).cpu().numpy()
                val_preds_ep.extend(preds)
                val_true_ep.extend(labels.cpu().numpy())

        val_f1  = f1_score(val_true_ep, val_preds_ep, average='macro', zero_division=0)
        val_acc = (np.array(val_preds_ep) == np.array(val_true_ep)).mean()
        n_tr    = len(train_loader)
        n_va    = len(val_loader)

        print(f'Epoch {epoch+1}/{epochs} | '
              f'Train Loss={epoch_loss/n_tr:.4f} | '
              f'Val Loss={val_loss_total/n_va:.4f} | '
              f'Val F1-Macro={val_f1:.4f} | Val Acc={val_acc:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1  = val_f1
            patience_ctr = 0
            torch.save(model.state_dict(), output_dir + '/model_best_m2.pt')
            print(f'  ✓ New best Val F1-Macro={best_val_f1:.4f} — saved.')
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'  Early stopping at epoch {epoch+1}')
                break

    with open(output_dir + '/m2_config.json', 'w') as f:
        json.dump({
            'base_model':  CFG.SENTIMENT_MODEL,
            'best_val_f1': best_val_f1,
            'n_classes':   2,
            'id2label':    {0: 'Negative', 1: 'Positive'},
        }, f, indent=2)

    print(f'M2 training done. Best Val F1-Macro: {best_val_f1:.4f}')

In [ ]:
def run_m2_calibration(m2_val_df, tokenizer, model, default_th=0.65):
    print('Computing M2 predictions on val set for calibration...')
    val_preds, val_probs = predict_sentiment_batch_raw(
        m2_val_df['sentence'].tolist(),
        m2_val_df['aspect'].tolist(),
        tokenizer, model,
    )
    val_true = m2_val_df['label_idx'].values
    val_conf = val_probs.max(axis=1)

    print(f'Val set: {len(val_preds)} aspect-pairs (Binary: Neg/Pos)')
    print(f'Baseline — Acc={(val_preds==val_true).mean():.4f} | '
          f'F1={f1_score(val_true, val_preds, average="macro", zero_division=0):.4f}')

    print(f'\n{classification_report(val_true, val_preds, labels=[0, 1], target_names=SENTIMENT_LABELS, zero_division=0)}')

    print('Confidence threshold calibration (Val, Neg/Pos):')
    print(f'  {"Threshold":>9} | {"Coverage%":>9} | {"F1-Macro":>8} | {"Accuracy":>8}')
    print('  ' + '-'*46)

    best_score, best_th = 0, default_th
    for th in np.arange(0.40, 0.92, 0.05):
        mask = val_conf >= th
        if mask.sum() < 10:
            break
        cov   = mask.mean() * 100
        f1_t  = f1_score(val_true[mask], val_preds[mask], average='macro', zero_division=0)
        acc_t = (val_preds[mask] == val_true[mask]).mean()
        flag  = ' ◀' if f1_t > best_score else ''
        print(f'  {th:>9.2f} | {cov:>9.1f} | {f1_t:>8.4f} | {acc_t:>8.4f}{flag}')
        if f1_t > best_score:
            best_score, best_th = f1_t, th

    print(f'\n→ CONF_THRESHOLD_M2 = {best_th:.2f}  (Val F1={best_score:.4f})')
    print('  Override manually trade off coverage / precision.')
    return best_th

In [ ]:
train_model2(
    train_df=m2_train_df, val_df=m2_val_df,
    tokenizer=m2_tokenizer, model=m2_model,
    class_weights=m2_class_weights,
)

#  Load best fine-tuned checkpoint
ckpt_path = CFG.M2_OUTPUT + '/model_best_m2.pt'
if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        f'Checkpoint does not exists {ckpt_path}. '
        'Please rerun the cell train or check the CFG.M2_OUTPUT.'
    )
m2_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
m2_model.eval()
print('M2 fine-tuned model loaded.')

CONF_THRESHOLD_M2 = run_m2_calibration(m2_val_df, m2_tokenizer, m2_model)

# Save threshold
with open(CFG.M2_OUTPUT + '/m2_threshold.json', 'w') as f:
    json.dump({'conf_threshold_m2': CONF_THRESHOLD_M2}, f, indent=2)
print(f'\nFinal CONF_THRESHOLD_M2 = {CONF_THRESHOLD_M2:.2f} saved.')

Epoch 1/10 | Train Loss=0.5051 | Val Loss=0.3277 | Val F1-Macro=0.9258 | Val Acc=0.9282
  ✓ New best Val F1-Macro=0.9258 — saved.
Epoch 2/10 | Train Loss=0.2887 | Val Loss=0.3833 | Val F1-Macro=0.8951 | Val Acc=0.8974
Epoch 3/10 | Train Loss=0.2500 | Val Loss=0.3544 | Val F1-Macro=0.9249 | Val Acc=0.9282
Epoch 4/10 | Train Loss=0.2322 | Val Loss=0.3597 | Val F1-Macro=0.9193 | Val Acc=0.9231
  Early stopping at epoch 4
M2 training done. Best Val F1-Macro: 0.9258
M2 fine-tuned model loaded.
Computing M2 predictions on val set for calibration...
Val set: 195 aspect-pairs (Binary: Neg/Pos)
Baseline — Acc=0.9282 | F1=0.9258

              precision    recall  f1-score   support

    Negative       0.88      0.95      0.91        77
    Positive       0.96      0.92      0.94       118

    accuracy                           0.93       195
   macro avg       0.92      0.93      0.93       195
weighted avg       0.93      0.93      0.93       195

Confidence threshold calibration (Val, Neg/Po

## 8. Evaluation on Global Test Set (open once)

In [ ]:
print('GLOBAL TEST SET EVALUATION — MODEL 1 (Aspect Detection)')
test_loader = DataLoader(ds_test, batch_size=CFG.M1_BATCH, shuffle=False, num_workers=2)
h1_probs_test, h1_labels_test, h2_probs_test, h2_labels_test, h3_probs_test, h3_labels_test = collect_predictions_m1(m1_model, test_loader)

# H1
h1_preds_test = (h1_probs_test >= THRESHOLD_H1).astype(int)
f1_h1_test = f1_score(h1_labels_test, h1_preds_test, average='macro', zero_division=0)
print(f'H1 (Related)   F1-Macro: {f1_h1_test:.4f}')

# H2
h2_preds_test = (h2_probs_test >= THRESHOLD_H2).astype(int)
f1_h2_test = f1_score(h2_labels_test, h2_preds_test, average='macro', zero_division=0)
print(f'H2 (Technical) F1-Macro: {f1_h2_test:.4f}')

thresh_arr = np.array([THRESHOLDS_H3[cls] for cls in ASPECT_CLASSES])
h3_preds_test = (h3_probs_test >= thresh_arr).astype(int)

print(f'\n{"Aspect":<22} | {"F1":>6}')
print('-' * 32)
for i, cls in enumerate(ASPECT_CLASSES):
    f1_i = f1_score(h3_labels_test[:, i], h3_preds_test[:, i], zero_division=0)
    print(f'{cls:<22} | {f1_i:.4f}')

f1_macro = f1_score(h3_labels_test, h3_preds_test, average='macro',   zero_division=0)
f1_samp  = f1_score(h3_labels_test, h3_preds_test, average='samples', zero_division=0)
print('-' * 32)
print(f'F1-Macro   : {f1_macro:.4f}')
print(f'F1-Samples : {f1_samp:.4f}')

GLOBAL TEST SET EVALUATION — MODEL 1 (Aspect Detection)
H1 (Related)   F1-Macro: 0.7780
H2 (Technical) F1-Macro: 0.8611

Aspect                 |     F1
--------------------------------
battery                | 0.8889
design & build         | 0.8108
display & audio        | 0.8163
input                  | 0.8485
performance            | 0.7755
price                  | 0.8519
software               | 0.7308
--------------------------------
F1-Macro   : 0.8175
F1-Samples : 0.8411


In [ ]:
print('GLOBAL TEST SET EVALUATION — MODEL 2 (Fine-tuned Sentiment)')
print('Evaluating on ALL per-aspect pairs (single + multi-aspect)\n')

eval_test_df = m2_test_df.rename(columns={'label_idx': 'true_label'}).copy()
n_tech_test  = (df_full_test[CFG.IS_TECH_COL] == 1).sum()
print(f'M2 Test pairs: {len(eval_test_df)} (từ {n_tech_test} technical rows)')

test_preds, test_probs = predict_sentiment_batch_raw(
    eval_test_df['sentence'].tolist(),
    eval_test_df['aspect'].tolist(),
    m2_tokenizer, m2_model,
)
test_true = eval_test_df['true_label'].values
test_conf = test_probs.max(axis=1)

print(classification_report(test_true, test_preds, labels=[0, 1], target_names=SENTIMENT_LABELS, zero_division=0))
print(f'F1-Macro : {f1_score(test_true, test_preds, average="macro", zero_division=0):.4f}')
print(f'Accuracy : {(test_preds == test_true).mean():.4f}')

# Confident-only subset
mask_conf = test_conf >= CONF_THRESHOLD_M2
if mask_conf.sum() > 0:
    f1_conf  = f1_score(test_true[mask_conf], test_preds[mask_conf],
                         average='macro', zero_division=0)
    acc_conf = (test_preds[mask_conf] == test_true[mask_conf]).mean()
    print(f'\nConfident subset (>={CONF_THRESHOLD_M2:.2f}): '
          f'{mask_conf.sum()}/{len(test_preds)} pairs ({mask_conf.mean()*100:.1f}%)')
    print(f'  F1-Macro : {f1_conf:.4f} | Accuracy : {acc_conf:.4f}')

# Per-aspect breakdown
eval_test_df = eval_test_df.copy()
eval_test_df['pred']    = test_preds
eval_test_df['correct'] = (test_preds == test_true)
eval_test_df['conf']    = test_conf

print(f'\nPer-aspect breakdown (test, all pairs):')
print(f'  {"Aspect":<20} | {"N":>4} | {"Acc":>6} | {"F1":>6} | {"Acc(conf)":>10}')
print('  ' + '-' * 58)

for asp in ASPECT_CLASSES:
    sub = eval_test_df[eval_test_df['aspect'] == asp]
    if len(sub) == 0:
        continue
    sub_conf = sub[sub['conf'] >= CONF_THRESHOLD_M2]
    acc_all  = sub['correct'].mean()
    f1_asp   = f1_score(sub['true_label'], sub['pred'], average='macro', zero_division=0)
    acc_c    = sub_conf['correct'].mean() if len(sub_conf) > 0 else float('nan')
    print(f'  {asp:<20} | {len(sub):>4} | {acc_all:>6.3f} | {f1_asp:>6.3f} | {acc_c:>10.3f}')

GLOBAL TEST SET EVALUATION — MODEL 2 (Fine-tuned Sentiment)
Evaluating on ALL per-aspect pairs (single + multi-aspect)

M2 Test pairs: 195 (từ 182 technical rows)
              precision    recall  f1-score   support

    Negative       0.85      0.95      0.89        75
    Positive       0.96      0.89      0.93       120

    accuracy                           0.91       195
   macro avg       0.90      0.92      0.91       195
weighted avg       0.92      0.91      0.91       195

F1-Macro : 0.9097
Accuracy : 0.9128

Confident subset (>=0.90): 148/195 pairs (75.9%)
  F1-Macro : 0.9591 | Accuracy : 0.9595

Per-aspect breakdown (test, all pairs):
  Aspect               |    N |    Acc |     F1 |  Acc(conf)
  ----------------------------------------------------------
  battery              |   12 |  1.000 |  1.000 |      1.000
  design & build       |   52 |  0.942 |  0.940 |      0.946
  display & audio      |   23 |  1.000 |  1.000 |      1.000
  input                |   15 |  0.667

In [ ]:
import shutil

# 1. Saved tokenizers together with the models
m1_tokenizer.save_pretrained(CFG.M1_OUTPUT)
print(f'✅ Saved M1 tokenizer at {CFG.M1_OUTPUT}')

m2_tokenizer.save_pretrained(CFG.M2_OUTPUT)
print(f'✅ Succesfully saved M2 tokenizer at {CFG.M2_OUTPUT}')

# 2. Saved models as zip file to easily download
shutil.make_archive('model1_multitask', 'zip', CFG.M1_OUTPUT)
print('✅ zipped Model 1 to model1_multitask.zip')

shutil.make_archive('model2_sentiment', 'zip', CFG.M2_OUTPUT)
print('✅ zipped Model 2 to model2_sentiment.zip')

print('\n🎉 Well done! Everything is saved on the side bar!')

✅ Saved M1 tokenizer at ./model1_multitask
✅ Succesfully saved M2 tokenizer at ./model2_sentiment
✅ zipped Model 1 to model1_multitask.zip
✅ zipped Model 2 to model2_sentiment.zip

🎉 Well done! Everything is saved on the side bar!
